# Análisis exploratorio: mercado de paneles solares en La Habana

**Proyecto:** Mercado solar Cuba — recopilación y análisis de anuncios públicos.
**Fuente de datos:** anuncios públicos de Revolico (o datos autorizados importados).
**Fecha de creación:** 2026.

Este notebook analiza el conjunto de datos limpio generado por el pipeline del
proyecto (`main.py import` / `main.py scrape`).

> ⚠️ **Advertencia sobre los datos.** Los precios corresponden a **anuncios
> publicados**, no necesariamente a ventas realizadas. No se convierten monedas
> (cada precio se analiza en su moneda original). Si el dataset contiene datos
> de ejemplo, son sintéticos y solo sirven para demostrar el flujo de trabajo.
>
> Se distingue entre **panel individual**, **varios paneles** y **sistema
> completo (kit)**: comparar sus precios directamente sería incorrecto.


In [ ]:
# --- Configuración e imports ---
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
    HAS_SEABORN = True
except ImportError:  # seaborn es opcional
    HAS_SEABORN = False
    plt.style.use("ggplot")

pd.set_option("display.max_columns", 50)

# Rutas relativas (funciona desde notebooks/ o desde la raíz del repo).
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
PROCESSED = ROOT / "data" / "processed" / "paneles_solares_habana_limpio.csv"
EXAMPLE = ROOT / "data" / "examples" / "ejemplo_paneles_habana.csv"
FIGDIR = ROOT / "output"
FIGDIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED if PROCESSED.exists() else EXAMPLE
print("Ejecutado desde :", CWD)
print("Leyendo datos de:", DATA_PATH)
print("seaborn:", HAS_SEABORN)


In [ ]:
# --- Carga de datos ---
df = pd.read_csv(DATA_PATH, dtype=str)

for col in ["precio_valor", "potencia_w", "vida_util_anios"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Filas:", len(df), "| Columnas:", len(df.columns))
df.head()


In [ ]:
# --- Aviso si los datos son de ejemplo (sintéticos) ---
if "fuente" in df.columns:
    fuentes = df["fuente"].fillna("(desconocido)").value_counts()
    print("Registros por fuente:")
    print(fuentes.to_string())
    if (df["fuente"].astype(str).str.lower() == "ejemplo").all():
        print("\nAVISO: el dataset actual son DATOS DE EJEMPLO (sintéticos). "
              "No representan el mercado real; las conclusiones son demostrativas.")


## 1. Cantidad total de anuncios recopilados

Se cuentan los registros únicos (ya deduplicados por URL/ID en el pipeline) y se
separa la oferta de **paneles** del resto de equipos (inversores, baterías, etc.),
porque no son comparables entre sí.


In [ ]:
total = len(df)
print(f"Total de anuncios únicos: {total}")

if "tipo_producto" in df.columns:
    print("\nAnuncios por tipo de producto:")
    print(df["tipo_producto"].fillna("(desconocido)").value_counts().to_string())

if "alcance" in df.columns:
    print("\nAnuncios por alcance del precio:")
    print(df["alcance"].fillna("(no determinado)").value_counts().to_string())


## 2. Distribución de anuncios por municipio

El municipio se normalizó a los 15 municipios de La Habana. Los anuncios cuya
ubicación no permite identificar un municipio con seguridad quedan como
`(desconocido)` y **no** se asignan a ciegas.


In [ ]:
if "municipio" in df.columns:
    mun = df["municipio"].fillna("(desconocido)").value_counts()
    print(mun.to_string())

    fig, ax = plt.subplots(figsize=(9, 6))
    mun.sort_values().plot(kind="barh", ax=ax, color="#1c7ed6")
    ax.set_title("Anuncios por municipio de La Habana")
    ax.set_xlabel("Cantidad de anuncios")
    ax.set_ylabel("Municipio")
    fig.tight_layout()
    fig.savefig(FIGDIR / "nb_anuncios_por_municipio.png", dpi=120)
    plt.show()
else:
    print("No hay columna 'municipio'.")


## 3. Distribución de precios anunciados

Se analiza **una moneda a la vez** (no se convierten divisas). Para evitar
mezclar un panel con un sistema completo, el histograma usa solo las ofertas
cuyo alcance es de panel (individual o varios paneles).


In [ ]:
# Filtro base: ofertas de paneles (no kits/sistemas) para comparar peras con peras.
def solo_paneles(frame):
    if "alcance" in frame.columns:
        return frame[frame["alcance"].isin(["panel individual", "varios paneles"])]
    return frame


paneles = solo_paneles(df)

if "moneda" in df.columns and "precio_valor" in df.columns:
    print("Registros por moneda (todos):")
    print(df["moneda"].fillna("(sin moneda)").value_counts().to_string())

    for moneda in df["moneda"].dropna().unique():
        subset = paneles[paneles["moneda"] == moneda]["precio_valor"].dropna()
        if subset.empty:
            continue
        print(f"\n--- Paneles en {moneda} (n={len(subset)}) ---")
        print(subset.describe().round(2).to_string())

    principal = paneles["moneda"].dropna().mode()
    principal = principal.iloc[0] if not principal.empty else None
    if principal is not None:
        subset = paneles[paneles["moneda"] == principal]["precio_valor"].dropna()
        if len(subset) >= 2:
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.hist(subset, bins=12, color="#1c7ed6", edgecolor="white")
            ax.set_title(f"Distribución de precios de paneles ({principal})")
            ax.set_xlabel(f"Precio ({principal})")
            ax.set_ylabel("Cantidad de anuncios")
            fig.tight_layout()
            fig.savefig(FIGDIR / "nb_distribucion_precios.png", dpi=120)
            plt.show()


## 4. Comparación de precios según la potencia nominal

Se relaciona la potencia del panel (W) con el precio, usando **solo paneles**
y una sola moneda. La potencia de un sistema (kit) no es la de un panel, por eso
los kits quedan fuera.


In [ ]:
if {"precio_valor", "potencia_w", "moneda"}.issubset(paneles.columns):
    principal = paneles["moneda"].dropna().mode()
    principal = principal.iloc[0] if not principal.empty else None
    sub = paneles[(paneles["moneda"] == principal)].dropna(subset=["precio_valor", "potencia_w"])
    print(f"Paneles con precio y potencia en {principal}: {len(sub)}")
    if len(sub) >= 3:
        corr = sub["potencia_w"].corr(sub["precio_valor"])
        print(f"Correlación de Pearson potencia-precio: {corr:.3f}")
        fig, ax = plt.subplots(figsize=(8, 5))
        if HAS_SEABORN:
            sns.regplot(data=sub, x="potencia_w", y="precio_valor", ax=ax,
                        scatter_kws={"alpha": 0.7}, line_kws={"color": "#e8590c"})
        else:
            ax.scatter(sub["potencia_w"], sub["precio_valor"], alpha=0.7)
        ax.set_title(f"Potencia vs. precio de paneles ({principal})")
        ax.set_xlabel("Potencia (W)")
        ax.set_ylabel(f"Precio ({principal})")
        fig.tight_layout()
        fig.savefig(FIGDIR / "nb_potencia_vs_precio.png", dpi=120)
        plt.show()
    else:
        print("No hay suficientes datos para esta comparación.")


## 5. Frecuencia de las marcas disponibles

La marca se deriva del texto **solo** cuando aparece una marca conocida; no se
deduce por parecido ni por suposición.


In [ ]:
if "marca" in df.columns:
    marcas = df["marca"].fillna("(sin marca identificada)").value_counts()
    print(marcas.to_string())
    if len(marcas) > 1:
        fig, ax = plt.subplots(figsize=(8, 5))
        marcas.sort_values().plot(kind="barh", ax=ax, color="#2f9e44")
        ax.set_title("Frecuencia de marcas identificadas")
        ax.set_xlabel("Cantidad de anuncios")
        ax.set_ylabel("Marca")
        fig.tight_layout()
        fig.savefig(FIGDIR / "nb_marcas.png", dpi=120)
        plt.show()


## 6. Frecuencia de los modelos identificados

El modelo se extrae de forma conservadora (marca + tesela con dígitos). Es
normal que muchos anuncios **no** permitan identificar el modelo.


In [ ]:
if "modelo" in df.columns:
    modelos = df["modelo"].fillna("(sin modelo identificado)").value_counts()
    print(modelos.to_string())


## 7. Distribución de las garantías anunciadas

Se conserva el texto original de la garantía (años o meses). Recordatorio: la
garantía **no** es lo mismo que la vida útil.


In [ ]:
if "garantia" in df.columns:
    garantias = df["garantia"].fillna("(sin garantía indicada)").value_counts()
    print(garantias.to_string())


## 8. Análisis de la vida útil declarada

Vida útil normalizada a años cuando el anuncio la indica explícitamente. No se
confunde con la garantía.


In [ ]:
if "vida_util_anios" in df.columns:
    vidas = df["vida_util_anios"].dropna()
    print(f"Anuncios con vida útil declarada: {len(vidas)} de {len(df)}")
    if not vidas.empty:
        print(vidas.describe().round(2).to_string())
        print("\nValores declarados:")
        print(df["vida_util"].dropna().value_counts().to_string())


## 9. Cantidad de anuncios con información técnica incompleta

No se descartan los anuncios incompletos: se documentan sus limitaciones.


In [ ]:
tecnicos = [c for c in ["marca", "modelo", "potencia_w", "vida_util", "garantia"] if c in df.columns]
resumen = []
for col in tecnicos:
    faltan = df[col].isna().sum() + (df[col].astype(str).str.strip() == "").sum()
    resumen.append({"campo": col, "faltantes": int(faltan),
                    "porcentaje": round(100 * faltan / len(df), 1) if len(df) else 0.0})
resumen = pd.DataFrame(resumen)
print(resumen.to_string(index=False))


## 10. Comparación de precios entre municipios (cuando hay datos suficientes)

Solo se comparan municipios con **al menos 3** anuncios de paneles en la misma
moneda. Con menos datos, las diferencias no son concluyentes.


In [ ]:
if {"municipio", "precio_valor", "moneda"}.issubset(paneles.columns):
    principal = paneles["moneda"].dropna().mode()
    principal = principal.iloc[0] if not principal.empty else None
    sub = paneles[paneles["moneda"] == principal].dropna(subset=["precio_valor", "municipio"])
    tabla = (sub.groupby("municipio")["precio_valor"]
                .agg(n="count", mediana="median", media="mean", min="min", max="max")
                .sort_values("mediana", ascending=False).round(2))
    tabla = tabla[tabla["n"] >= 3]
    if tabla.empty:
        print("No hay municipios con >= 3 anuncios de paneles para comparar de forma fiable.")
    else:
        print(f"Precios de paneles en {principal} por municipio (n>=3):")
        print(tabla.to_string())


## Conclusiones y limitaciones

- Los datos provienen de **anuncios publicados**, no de transacciones cerradas:
  los precios son ofertas, no ventas confirmadas.
- No se convierten monedas; los precios en CUP y USD se analizan por separado.
- No se comparan paneles individuales con kits/sistemas completos.
- Las marcas, modelos, garantías y vidas útiles son **declaraciones del
  vendedor**, no características verificadas de forma independiente.
- Los municipios se normalizaron a los 15 de La Habana; los casos ambiguos
  quedan como desconocidos.
- El tamaño de la muestra condiciona la fiabilidad: pocos anuncios por municipio
  impiden conclusiones firmes.
